# Features

Cloud cover and cloud height from EUMetSat data archive spanning 2024 year.

In [28]:
import matplotlib.pyplot as plt
import numpy as np
import math as maths
import torch, torch.nn as nn
import pandas as pd

torch.cuda.is_available()

True

In [29]:
sat_data = np.load("data/ireland_clouds_20240101_20250101.npz")
print(sat_data.files)

['times', 'cloud_mask', 'cth_km', 'lat', 'lon']


In [30]:
print(sat_data["cloud_mask"].shape)
print(sat_data["cloud_mask"][1, :, :])

(8784, 90, 120)
[[2 2 2 ... 2 2 2]
 [2 2 2 ... 2 2 2]
 [2 2 2 ... 2 2 2]
 ...
 [2 2 2 ... 2 2 2]
 [2 0 0 ... 2 2 2]
 [0 0 0 ... 2 2 2]]


In [31]:
csv_name = ["hly532.csv","hly3904_subset.csv", "hly4935_subset.csv", "hly518_subset.csv"]

dates = []

def get_target_data(csv_name):
    data = pd.read_csv(csv_name,skiprows=41, low_memory=False)
    data = data.rename(columns={'ind': 'Rainfall_indicator', "ind.1": "Dry_bulb_indicator", "ind.2": "Wet_bulb_indicator", "ind.3": "wind_speed_indicator", "ind.4": "wind_dir_indicator"})

    data["date"] = pd.to_datetime(data["date"], format="%d-%b-%Y %H:%M")
    mask = data["date"].dt.year == 2024
    data = data[mask]
    target = data["temp"].values
    print(f"{csv_name} target shape: {target.shape}")
    dates.append(data["date"].values)

    return target




target_data = []

for csv in csv_name:
    target_data.append(get_target_data(csv))

dates = np.array(dates).T
print(f"dates shape: {dates.shape}")

for date in dates:
    assert np.all(date[0] == date[1 :]), "Dates are not the same across all CSV files."



hly532.csv target shape: (8784,)
hly3904_subset.csv target shape: (8784,)
hly4935_subset.csv target shape: (8784,)
hly518_subset.csv target shape: (8784,)
dates shape: (8784, 4)


In [32]:
target_data = np.array(target_data)
print(f"target_data shape: {target_data.shape}")


target_data shape: (4, 8784)


In [ ]:
cloud_mask = sat_data["cloud_mask"]
cth_km = sat_data["cth_km"]
sat_times = sat_data["times"]

# station rows must be the same hours in the same order as the satellite frames (all UTC)
assert np.array_equal(dates[:, 0].astype("datetime64[us]"), sat_times)

# one True/False per HOUR: missing only if the WHOLE frame is "no data" (3)
sat_ok = ~(cloud_mask == 3).all(axis=(1, 2))                 # (8784,)

# target_data is (stations, hours): an hour is usable only if EVERY station has a temp,
# so collapse the station axis -> one value per hour
target_ok = ~np.isnan(target_data).any(axis=0)               # (8784,)

hour_ok = sat_ok & target_ok
print(f"hours: {len(hour_ok)}, missing: {(~hour_ok).sum()}")
print("missing hours:", sat_times[~hour_ok])

In [ ]:
K = 6   # hours of satellite history per sample
H = 3   # predict temp this many hours after the last input frame

# Don't delete the missing rows from the arrays -- that would join hours either side of a
# gap into one "continuous" sequence. Keep the full hourly arrays and instead list the
# samples that are valid: a sample ending at hour t uses frames t-K+1..t and target t+H,
# and is kept only if all of those hours are present.
sample_t = np.array([t for t in range(K - 1, len(hour_ok) - H)
                     if hour_ok[t - K + 1:t + 1].all() and hour_ok[t + H]])

print(f"possible samples: {len(hour_ok) - H - K + 1}, kept: {len(sample_t)}, "
      f"dropped: {len(hour_ok) - H - K + 1 - len(sample_t)}")

# e.g. inputs and target for the first valid sample
t = sample_t[0]
X0 = cloud_mask[t - K + 1:t + 1]    # (K, 90, 120)
y0 = target_data[:, t + H]          # (4,) one temp per station
print(X0.shape, y0)